#Initialization

In [0]:
import pyspark.sql.functions as F

#Read Bronze Table

In [0]:
doctors = spark.table("health.bronze.doctors")

#Data Understanding

###Review Schema

In [0]:
doctors.printSchema()
display(doctors.limit(10))

#Data Profiling

###Check Null Values

In [0]:
display(
    doctors.select(
        [
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in doctors.columns
        ]
    )
)

###Whitespace Validation

In [0]:
string_cols = [
    field.name
    for field in doctors.schema.fields
    if field.dataType.simpleString() == "string"
]

display(
    doctors.select(
        [
            F.sum(
                F.when(
                    F.col(c) != F.trim(F.col(c)),
                    1
                ).otherwise(0)
            ).alias(c)
            for c in string_cols
        ]
    )
)

###Business Key Validation

In [0]:
total_rows = doctors.count()

distinct_doctor_ids = (
    doctors
    .select("doctor_id")
    .distinct()
    .count()
)

print(f"Total Rows: {total_rows}")
print(f"Distinct Doctor IDs: {distinct_doctor_ids}")

###Specialty Validation

In [0]:
display(
    doctors.groupBy("specialty")
    .count()
    .orderBy("specialty")
)

###Duplicate Validation

In [0]:
print(f"Total Rows: {doctors.count()}")

print(
    f"Rows After dropDuplicates(): {doctors.dropDuplicates().count()}"
)

#Transformation

###Remove Duplicate Records

In [0]:
doctors_clean = doctors.dropDuplicates()

###Validation Summary

In [0]:
print(f"Valid Doctors: {doctors_clean.count()}")
print(f"Total Doctors: {doctors.count()}")

#Write Silver Table

In [0]:
(
    doctors_clean.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("health.silver.doctors")
)

###Verify Silver Table

In [0]:
display(
    spark.table("health.silver.doctors")
    .limit(10)
)